In [1]:
!pip install skfeature-chappers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.3/66.3 kB 2.1 MB/s eta 0:00:00


In [2]:
import sys
sys.path.append("/kaggle/input/library-metrics")

In [3]:
import numpy as np
import pandas as pd
import psutil, time, threading, tracemalloc
from skfeature.function.information_theoretical_based import FCBF
from metric_toolkit import (
    measure_time_and_memory,
    compute_energy, compute_carbon, compute_edp
)

# ============================================================
# 1️⃣ DỮ LIỆU
# ============================================================
X_train = pd.read_csv('/kaggle/input/data-time-complexity/X_train.csv')
y_train = pd.read_csv('/kaggle/input/data-time-complexity/y_train.csv').values.ravel()

# ============================================================
# 2️⃣ HÀM FEATURE SELECTION
# ============================================================
def run_fcbf(k=100):
    selected_indices = FCBF.fcbf(X_train.values, y_train)
    selected_indices = selected_indices[:k]

    mask = np.zeros(X_train.shape[1], dtype=bool)
    mask[selected_indices] = True

    num_evals = X_train.shape[1]
    X_sel = X_train.iloc[:, selected_indices]
    return mask, num_evals, X_sel

# ============================================================
# 3️⃣ LOGGER TRACEMALLOC (Python heap)
# ============================================================
mem_trace_log = []
logging_active = False

def log_tracemalloc(interval=0.1, max_samples=1000):
    start_time = time.time()
    count = 0
    while logging_active and count < max_samples:
        current, peak = tracemalloc.get_traced_memory()
        t = time.time() - start_time
        mem_trace_log.append((t, current / 1024**2, peak / 1024**2))
        time.sleep(interval)
        count += 1

# ============================================================
# 4️⃣ ĐO HIỆU NĂNG (có timeline memory)
# ============================================================
def measure_time_and_memory_with_timeline(func, *args, **kwargs):
    proc = psutil.Process()
    tracemalloc.start()
    start_time = time.time()

    # start logging
    global logging_active
    logging_active = True
    logger_thread = threading.Thread(target=log_tracemalloc, args=(0.1,))
    logger_thread.start()

    result = func(*args, **kwargs)

    # stop logging
    logging_active = False
    logger_thread.join()

    wall_time = time.time() - start_time
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    cpu_util = psutil.cpu_percent(interval=None)
    peak_mem_mb = peak_mem / 1024**2
    return {
        "result": result,
        "WallTime(s)": wall_time,
        "CPUUtil_Avg(%)": cpu_util,
        "PeakMem(MB)": peak_mem_mb,
    }

# ============================================================
# 5️⃣ CHẠY FCBF + GHI LOG
# ============================================================
timing = measure_time_and_memory_with_timeline(run_fcbf)
mask, num_evals, X_sel = timing["result"]

wall_time = timing["WallTime(s)"]
cpu_util = timing["CPUUtil_Avg(%)"]
peak_mem = timing["PeakMem(MB)"]
base_mem = psutil.Process().memory_info().rss / 1024**2

energy = compute_energy(cpu_util, wall_time)
carbon = compute_carbon(energy)
edp = compute_edp(energy, wall_time)

# ============================================================
# 6️⃣ LƯU FILE KẾT QUẢ
# ============================================================
fs_metrics = {
    "FS_Method": "FCBF",
    "NumFeatures": int(mask.sum()),
    "NumEvals": num_evals,
    "WallTime(s)": wall_time,
    "CPUUtil(%)": cpu_util,
    "PeakMem(MB)": peak_mem,
    "BaseMem(MB)": base_mem,
    "Energy(J)": energy,
    "Carbon(gCO2e)": carbon,
    "EDP(J*s)": edp,
    "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}

np.save("/kaggle/working/fcbf_mask.npy", mask)
pd.DataFrame([fs_metrics]).to_csv("/kaggle/working/fs_metrics_fcbf.csv", index=False)

# Tracemalloc timeline
mem_df = pd.DataFrame(mem_trace_log, columns=["Time(s)", "Current(MB)", "Peak(MB)"])
mem_df.to_csv("/kaggle/working/tracemalloc_timeline_fcbf.csv", index=False)

# ============================================================
# 7️⃣ IN RA KẾT QUẢ
# ============================================================
print("✅ FCBF Feature Selection done (no classifier).")
print(f"→ Selected features: {mask.sum()}")
print(f"→ WallTime: {wall_time:.3f}s | CPU Avg: {cpu_util:.2f}%")
print(f"→ PeakMem (Python heap): {peak_mem:.2f} MB")
print("→ Saved: fs_metrics_fcbf.csv, tracemalloc_timeline_fcbf.csv, fcbf_mask.npy")


✅ FCBF Feature Selection done (no classifier).
→ Selected features: 100
→ WallTime: 491.917s | CPU Avg: 25.50%
→ PeakMem (Python heap): 12.50 MB
→ Saved: fs_metrics_fcbf.csv, tracemalloc_timeline_fcbf.csv, fcbf_mask.npy
